## Audit ARF's existing 414 EVNT-typed entities

In [2]:
import pandas as pd

df = pd.read_parquet("../data/arf_chunks_parsed.parquet")
print(df.dtypes)
print(type(df["relations"].iloc[0]))
sample_val = df["relations"].iloc[0]
print(sample_val[:300] if isinstance(sample_val, str) else sample_val[:2])

book_id                 str
title                   str
author                  str
author_gender           str
author_birth_year       str
author_death_year       str
release_date            str
pg_subjects             str
topics                  str
chunk_id                str
chunk                   str
relations               str
relations_parsed     object
dtype: object
<class 'str'>
[]


In [3]:
import ast
from collections import Counter

def parse_relations(val):
    if val is None:
        return []
    if isinstance(val, str):
        return ast.literal_eval(val)
    return list(val)  # already parsed — adjust based on Cell 0's finding

evnt_rows = []
for _, row in df.iterrows():
    for r in parse_relations(row["relations"]):
        if r.get("entity1Type") == "EVNT":
            evnt_rows.append({"book_id": row["book_id"], "raw_name": r["entity1"],
                               "chunk_id": row["chunk_id"], "relation": r["relation"],
                               "other_type": r.get("entity2Type"), "other_entity": r.get("entity2")})
        if r.get("entity2Type") == "EVNT":
            evnt_rows.append({"book_id": row["book_id"], "raw_name": r["entity2"],
                               "chunk_id": row["chunk_id"], "relation": r["relation"],
                               "other_type": r.get("entity1Type"), "other_entity": r.get("entity1")})

evnt_df = pd.DataFrame(evnt_rows)

# 1. Reconcile with README's "414" figure — is that unique entities or
#    total occurrences? These imply very different things about how much
#    distinct event vocabulary actually exists.
print(f"EVNT entity-slot occurrences: {len(evnt_df)}")
print(f"Unique raw EVNT strings: {evnt_df['raw_name'].nunique()}")

# 2. Coverage — concentrated in a couple of books, or broadly usable?
per_book = evnt_df.groupby("book_id")["raw_name"].nunique().sort_values(ascending=False)
print(f"\nBooks with >=1 EVNT entity: {evnt_df['book_id'].nunique()} / {df['book_id'].nunique()}")
print(per_book.head(10))

EVNT entity-slot occurrences: 414
Unique raw EVNT strings: 331

Books with >=1 EVNT entity: 74 / 96
book_id
22066    24
32543    12
34025    12
44       12
9913     12
30365    11
35671    11
6941     11
6053     10
74593    10
Name: raw_name, dtype: int64


In [4]:
# 3. Specific occurrences ("the second battle") vs generic recurring
#    concepts ("war", "Christmas")? This is the single biggest factor —
#    a node representing a *type* isn't removable in the way v2's
#    "what if X died in event Y" needs.
uniq = evnt_df["raw_name"].drop_duplicates()
sample = uniq.sample(min(40, len(uniq)), random_state=42)
print("Sample raw EVNT strings (first 20):")
for s in sorted(sample)[:20]:
    print(f"  - {s!r}")

# 4. What's on the OTHER side of an EVNT relation? Mostly PER entities
#    would mean real "who was involved" signal already exists in ARF,
#    without inventing anything new.
print("\nEntity types connected to EVNT entities:")
print(evnt_df["other_type"].value_counts())

evnt_evnt_rows = []
for _, row in df.iterrows():
    for r in parse_relations(row["relations"]):
        if r.get("entity1Type") == "EVNT" and r.get("entity2Type") == "EVNT":
            evnt_evnt_rows.append({
                "book_id": row["book_id"], "chunk_id": row["chunk_id"],
                "entity1": r["entity1"], "relation": r["relation"], "entity2": r["entity2"],
            })

print(f"EVNT-to-EVNT relation instances (deduplicated, direction preserved): {len(evnt_evnt_rows)}")
for r in evnt_evnt_rows[:20]:
    print(f"  {r['entity1']!r} --{r['relation']}--> {r['entity2']!r}  (book {r['book_id']})")

Sample raw EVNT strings (first 20):
  - 'A Roman Singer'
  - 'Bacchae of Euripides'
  - "Bernard Peixada's murder"
  - 'Bull Run'
  - 'CHAPTER VI'
  - 'Epsom Spring Meeting'
  - "Ferguson's Idea"
  - 'Japanese earthquakes'
  - 'Little Barefoot'
  - 'Naperville game'
  - 'O, Promise Me!'
  - 'Shiloh'
  - 'Spanish Armada'
  - 'This'
  - 'Tri-Meet'
  - 'Tristan and Isolde'
  - 'Wedding Party'
  - 'arrest'
  - 'battle of Zama'
  - 'breakfast'

Entity types connected to EVNT entities:
other_type
PER     278
LOC      39
WTHR     27
ORG      26
FAC      12
TIME     11
CNCP     11
EVNT      6
OBJ       2
SENT      1
VEH       1
Name: count, dtype: int64
EVNT-to-EVNT relation instances (deduplicated, direction preserved): 3
  'Second Manassas' --associated_with--> 'big battle'  (book 22066)
  'dance and general feast' --associated_with--> 'slaying an elephant'  (book 32543)
  'Mons retreat' --occurs_in--> 'Battle of the Marne'  (book 35671)


In [5]:
# Pull a bigger, non-overlapping sample to see if the same category mix holds
sample2 = uniq.drop(sample.index).sample(min(60, len(uniq) - len(sample)), random_state=7)
for s in sorted(sample2):
    print(f"  - {s!r}")

  - 'Crismas festerval'
  - 'Euphrasia'
  - 'Exposition at Rio de Janeiro'
  - 'Fourth of July'
  - 'Grand Council'
  - 'Kentucky Derby'
  - 'Loan Exhibition'
  - "Molly's Story"
  - 'Monocacy raid'
  - 'Mons retreat'
  - 'Napoleonic wars'
  - 'Scheme'
  - 'Second Manassas'
  - 'Song'
  - 'Venus in Furs'
  - 'Vespers'
  - 'Zulu war'
  - 'a speech of mine'
  - 'affair'
  - 'assembly'
  - 'attacked'
  - 'battle on Lake Champlain'
  - 'bereavement'
  - 'case'
  - 'contest'
  - 'cricket match'
  - 'crime'
  - 'dance and general feast'
  - 'dinner'
  - 'disbanded'
  - 'entertainments'
  - "father's death"
  - 'festival in Leipsic'
  - 'forthcoming race with Oxford College'
  - 'his death'
  - 'his first marriage'
  - 'his intrigue'
  - 'his voyage'
  - 'home run'
  - 'invasion'
  - 'my lessons'
  - 'offer'
  - 'our late struggle in America'
  - 'our own Revolution'
  - 'our union'
  - 'sacrifices'
  - 'semi-final'
  - 'stirring and terrible events'
  - 'storm'
  - 'that disease which change

In [6]:
for _, row in df.iterrows():
    for r in parse_relations(row["relations"]):
        if r.get("entity1") == "Tristan and Isolde" or r.get("entity2") == "Tristan and Isolde":
            print(row["book_id"], row["chunk_id"], r)

47139 14 {'entity1': 'Godfried of Strasburg', 'entity2': 'Tristan and Isolde', 'entity1Type': 'PER', 'entity2Type': 'EVNT', 'relation': 'author_of'}
47139 14 {'entity1': 'Wagner', 'entity2': 'Tristan and Isolde', 'entity1Type': 'PER', 'entity2Type': 'EVNT', 'relation': 'author_of'}
47139 9 {'entity1': 'Wagner', 'entity2': 'Tristan and Isolde', 'entity1Type': 'PER', 'entity2Type': 'OBJ', 'relation': 'owns'}


In [7]:
# For each raw entity string, does it ever appear with more than one entity type?
type_records = []
for _, row in df.iterrows():
    for r in parse_relations(row["relations"]):
        type_records.append((r["entity1"], r.get("entity1Type")))
        type_records.append((r["entity2"], r.get("entity2Type")))

type_df = pd.DataFrame(type_records, columns=["name", "type"]).dropna()
inconsistent = type_df.groupby("name")["type"].nunique()
inconsistent = inconsistent[inconsistent > 1]
print(f"{len(inconsistent)} entity strings tagged with more than one type")
print(inconsistent.sort_values(ascending=False).head(20))

1119 entity strings tagged with more than one type
name
it               6
Virginia         6
Nadia            5
her              5
fire             5
Dundee           5
John             5
Beckford         4
Venus in Furs    4
Stonewall        4
Maple Ridge      4
Merrimac         4
Congress         4
Justice          4
Electra          4
Wellington       4
Tigris           4
Moon             4
Jamestown        4
Cumberland       4
Name: type, dtype: int64


In [8]:
# Does the relation TYPE itself already carry event-like signal,
# independent of the (now confirmed unreliable) EVNT entity tag?

all_relations = []
for _, row in df.iterrows():
    for r in parse_relations(row["relations"]):
        all_relations.append({
            "relation": r["relation"],
            "e1_type": r.get("entity1Type"),
            "e2_type": r.get("entity2Type"),
        })
rel_df = pd.DataFrame(all_relations)

print("Top 30 relation types, corpus-wide:")
print(rel_df["relation"].value_counts().head(30))

# Restrict to instances where NEITHER endpoint is EVNT-typed — these
# are candidate "hidden" events living purely in the relation label,
# on otherwise ordinary entity pairs (e.g. PER --killed_by--> PER).
no_evnt = rel_df[(rel_df["e1_type"] != "EVNT") & (rel_df["e2_type"] != "EVNT")]
print(f"\nRelation instances with no EVNT-typed endpoint: {len(no_evnt)} / {len(rel_df)}")
print("\nTop 30 relation types among these:")
print(no_evnt["relation"].value_counts().head(30))

Top 30 relation types, corpus-wide:
relation
companion_of        37249
relative_of         11894
child_of             7649
lover_of             5936
friend_of            5903
sibling_of           5628
spouse_of            5579
enemy_of             5370
rival_of             4842
parent_father_of     3816
parent_mother_of     2895
protector_of         2740
travel_to            2588
leader_of            2534
employer_of          2264
located_in           2223
lives_in             1880
member_of            1627
visits               1426
owns                 1288
mentor_of             946
part_of               831
believes_in           669
embodies              643
associated_with       590
occupied_by           535
owned_by              473
lived_in              466
travels_by            432
teacher_of            407
Name: count, dtype: int64

Relation instances with no EVNT-typed endpoint: 127920 / 128331

Top 30 relation types among these:
relation
companion_of        37249
relative_of  

In [9]:
print("Full relation type distribution, all 48 canonical types:")
print(rel_df["relation"].value_counts().to_string())

candidates = ["killed_by", "kills", "married_to", "marries", "attacked_by",
              "attacks", "captured_by", "captures", "rescued_by", "rescues",
              "died", "born", "fought", "wounded_by", "wounds"]
present = [c for c in candidates if c in rel_df["relation"].unique()]
print(f"\nEvent-shaped candidates actually present: {present}")
for c in present:
    print(f"  {c}: {(rel_df['relation'] == c).sum()} instances")

Full relation type distribution, all 48 canonical types:
relation
companion_of                                           37249
relative_of                                            11894
child_of                                                7649
lover_of                                                5936
friend_of                                               5903
sibling_of                                              5628
spouse_of                                               5579
enemy_of                                                5370
rival_of                                                4842
parent_father_of                                        3816
parent_mother_of                                        2895
protector_of                                            2740
travel_to                                               2588
leader_of                                               2534
employer_of                                             2264
located_in         

In [11]:
event_shaped = ["kills", "killed_by", "married_to", "attacks", "captures", "captured_by"]
sample = rel_df_full = []  # use the full row-level relations, not just the type-count df
for _, row in df.iterrows():
    for r in parse_relations(row["relations"]):
        if r["relation"] in event_shaped:
            sample.append({"book_id": row["book_id"], "chunk_id": row["chunk_id"],
                            "e1": r["entity1"], "e1_type": r.get("entity1Type"),
                            "relation": r["relation"],
                            "e2": r["entity2"], "e2_type": r.get("entity2Type")})

pd.DataFrame(sample).sort_values(["relation", "book_id", "chunk_id"])

,book_id,chunk_id,e1,e1_type,relation,e2,e2_type
5,22066,1377,Tyler's regiments,PER,attacks,the wall,FAC
8,22066,4964,2nd Corps,ORG,attacks,Chancellorsville,FAC
9,22066,593,McDowell,PER,attacks,Henry Hill,FAC
12,32543,358,Zulus,PER,attacks,settlement,FAC
13,34025,122,Hiero,PER,attacks,Messina,LOC
15,34025,467,Mithridates,PER,attacks,Cyzicus,LOC
17,37251,616,Sir John,PER,attacks,mansion or castle of Balquhain,FAC
18,37251,616,Sir Andrew Leslie,PER,attacks,lands of the Forbeses,LOC
21,45517,816,Company B,ORG,attacks,the snow fort,FAC
22,47139,1050,King Mark,PER,attacks,castle gate,FAC


Cleanup design decided but deferred: tiered event_candidate map
(mechanical exclude / flagged / needs_review), non-destructive,
mirrors the Week 4 resolution_map shape. Blocked on entity-resolution
Stage A/B (needs canonicalized names, not raw ARF strings) and a
Phase 4 book shortlist.